In [4]:
from langchain_ollama import ChatOllama
from dotenv import load_dotenv

load_dotenv()
code_act_mistral = ChatOllama(model="xingyaow/codeact-agent-mistral", temperature=0, max_tokens=2048)

In [ ]:
from jitgen.prebuilt.python import create_python_jitgen_v2
from jitgen_langchain.parser import JITGenParser

SYSTEM_PROMPT = """You are a helpful assistant assigned with the task of problem-solving. To achieve this,
you will be using an interactive coding environment equipped with a variety of tool
functions to assist you throughout the process.
At each turn, you should first provide your step-by-step thinking for solving the task.
Your thought process should be enclosed using "<thought>" tag, for example: <thought>
I need to print "Hello World!" </thought>.
After that, you have two options:
1) Interact with a Python programming environment and receive the corresponding output.
Your code should be enclosed using "<execute>" tag, for example: <execute> print("
Hello World!") </execute>.
2) Directly provide a solution that adheres to the required format for the given task.
Your solution should be enclosed using "<solution>" tag, for example: The answer is <
solution> A </solution>.
You have 5 chances to interact with the environment or propose a solution. You can only
propose a solution 2 times.
"""

async def stream_code_act_mistral(prompt: str):
    jitgen =create_python_jitgen_v2()
    
    messages =  [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    
    llm_stream = code_act_mistral.astream(messages)

    thought = ""
    observation = ""
    in_block = False
    last_two = []
    
    async for chunk in llm_stream:
        text = chunk.text
        thought += text
        if len(last_two) == 2:
            last_two.pop(0)
        last_two.append(text)
        
        joined_last_two = "".join(last_two)
        
        # print(f"Received chunk: {text}")
        if text:
            if "<execute>" in joined_last_two:
                in_block = True
                continue
            if "</execute>" in joined_last_two:
                in_block = False
                continue
            
            if in_block:
                observation += await jitgen.apush(text)
    
        yield text
    
    observation += await jitgen.aflush()
    yield observation
    
    messages.append({"role": "ai", "content": thought})
    messages.append({"role": "user", "content": "Observation: " + observation})

    llm_stream = code_act_mistral.astream(messages)
    thought = ""
    observation = ""
    in_block = False
    last_two = []
    
    async for chunk in llm_stream:
        text = chunk.text
        thought += text
        if len(last_two) == 2:
            last_two.pop(0)
        last_two.append(text)
        
        joined_last_two = "".join(last_two)
        
        # print(f"Received chunk: {text}")
        if text:
            if "<execute>" in joined_last_two:
                in_block = True
                continue
            if "</execute>" in joined_last_two:
                in_block = False
                continue
            
            if in_block:
                observation += await jitgen.apush(text)
    
        yield text
    
    observation += await jitgen.aflush()
    yield observation
    
    

In [24]:
async for chunk in stream_code_act_mistral("Print minimum sum of factors of 120."):
    print(chunk, end="")

To find the minimum sum of factors of 120, we need to first determine the factors of 120 and then find their sum.
<execute
factors = []
for i in range(1, 121):
    if 120 % i == 0:
        factors.append(i)
print(factors)
[1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, 24, 30, 40, 60, 120]
Now that we have the factors of 120, let's find their sum.
<execute
sum_of_factors = 0
for factor in factors:
    sum_of_factors += factor
print(sum_of_factors)

ValueError: Error detected. Halting further processing. name 'factors' is not defined